# Ultrasound Image Preprocessing Pipeline

Processes raw ultrasound images from multiple source directories:
1. Loads each image as grayscale (JPG / PNG / DICOM / DICOM-in-ZIP / MHA / NPY)
2. Normalises intensity (1st–99th percentile clip → [0, 1])
3. Optionally applies a trained **conus mask** — zeros out everything outside the imaging cone
4. Saves as 8-bit PNG at **native resolution** to `OUTPUT_DIR`
5. Writes `manifest.csv` (source path, output path, format, original size, conus coverage %)

Re-running is safe — already-processed files are skipped (`SKIP_EXISTING = True`).

In [1]:
!ls /content

drive  image_clenup  sample_data


In [2]:
import logging
import os
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().resolve().parents[0])  # one level up from image_clenup/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)

from image_clenup.pipeline import run_pipeline

print(f"Project root: {PROJECT_ROOT}")

Project root: /


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

In [15]:
DRIVE = "/content/drive/MyDrive/SHAD/project"

# ── Sources — add / remove paths as needed ────────────────────────────────────
SOURCE_PATHS = [
    # f"{DRIVE}/fetal_planes_db/Images",
    # f"{DRIVE}/fetal_head_circumference/training_set/training_set",
    # f"{DRIVE}/focus/validation/images",
    # f"{DRIVE}/FetalAbdominalSegmentationDataset/IMAGES",
    f"{DRIVE}/PSFHS/PSFHS/image_mha",
    # f"{DRIVE}/clinical_ultrasound_data",   # contains .zip archives with DICOMs
]

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = f"{DRIVE}/PSFHS/PSFHS/image_mha_preprocessed"

# ── Conus masking (optional) ──────────────────────────────────────────────────
# Set to None to skip masking (just normalise and save).
CONUS_CHECKPOINT = f"{DRIVE}/checkpoints/conus/best_conus.pt"
# CONUS_CHECKPOINT = None

# ── Performance ───────────────────────────────────────────────────────────────
WORKERS        = 8    # I/O threads for loading images in parallel
BATCH_SIZE_GPU = 32   # images per GPU forward pass for conus model
SKIP_EXISTING  = True # skip files already present in manifest.csv

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Sources: {len(SOURCE_PATHS)}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Conus checkpoint: {CONUS_CHECKPOINT}")

Device: cuda
Sources: 1
Output:  /content/drive/MyDrive/SHAD/project/PSFHS/PSFHS/image_mha_preprocessed
Conus checkpoint: /content/drive/MyDrive/SHAD/project/checkpoints/conus/best_conus.pt


## Run

In [20]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 51.4 MB/s eta 0:00:00:00:0100:01


In [21]:
manifest_path = run_pipeline(
    source_paths=SOURCE_PATHS,
    output_dir=OUTPUT_DIR,
    conus_checkpoint=CONUS_CHECKPOINT,
    workers=WORKERS,
    batch_size_gpu=BATCH_SIZE_GPU,
    skip_existing=SKIP_EXISTING,
    device=DEVICE,
)
print(f"Manifest written to: {manifest_path}")

  Processed 1358/1358...
Saved 1358 images to /content/drive/MyDrive/SHAD/project/PSFHS/PSFHS/image_mha_preprocessed
Manifest: /content/drive/MyDrive/SHAD/project/PSFHS/PSFHS/image_mha_preprocessed/manifest.csv
Manifest written to: /content/drive/MyDrive/SHAD/project/PSFHS/PSFHS/image_mha_preprocessed/manifest.csv


## Inspect results

In [17]:
import csv
from collections import Counter

with open(manifest_path, newline="") as f:
    rows = list(csv.DictReader(f))

ok     = [r for r in rows if r["status"] == "ok"]
errors = [r for r in rows if r["status"] != "ok"]

print(f"Total entries : {len(rows)}")
print(f"OK            : {len(ok)}")
print(f"Errors        : {len(errors)}")
print()
print("By format:")
for fmt, n in sorted(Counter(r["format"] for r in ok).items()):
    print(f"  {fmt:<12} {n}")

if errors:
    print()
    print("Error statuses:")
    for s, n in Counter(r["status"] for r in errors).items():
        print(f"  {s}: {n}")

Total entries : 1358
OK            : 0
Errors        : 1358

By format:

Error statuses:
  load_error: 1358


In [18]:
import matplotlib.pyplot as plt
import numpy as np

coverages = [float(r["conus_coverage_pct"])
             for r in ok if r["conus_coverage_pct"] != ""]

if coverages:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(coverages, bins=40, edgecolor="white", linewidth=0.4)
    axes[0].set_xlabel("Conus coverage (% of pixels)")
    axes[0].set_ylabel("Images")
    axes[0].set_title("Conus coverage distribution")
    axes[0].axvline(10, color="red", linestyle="--", label="<10% suspect")
    axes[0].legend()

    suspect = [r for r in ok
               if r["conus_coverage_pct"] != "" and float(r["conus_coverage_pct"]) < 10]
    axes[1].bar(["OK", "Suspect (<10%)"], [len(ok) - len(suspect), len(suspect)],
                color=["steelblue", "tomato"])
    axes[1].set_title("Image quality flag")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.show()
    print(f"Mean coverage: {np.mean(coverages):.1f}%  "
          f"Median: {np.median(coverages):.1f}%  "
          f"Suspect: {len(suspect)}")
else:
    print("No conus coverage data (pipeline ran without a conus checkpoint).")

No conus coverage data (pipeline ran without a conus checkpoint).


In [19]:
import random
from PIL import Image

N_SHOW = 12
COLS   = 4

sample_rows = random.sample(ok, min(N_SHOW, len(ok)))
rows_grid   = (len(sample_rows) + COLS - 1) // COLS

fig, axes = plt.subplots(rows_grid, COLS, figsize=(COLS * 3, rows_grid * 3))
axes = np.array(axes).reshape(-1)

for ax, row in zip(axes, sample_rows):
    img = Image.open(row["dst_path"])
    ax.imshow(img, cmap="gray")
    cov = row["conus_coverage_pct"]
    cov_str = f"  cov={cov}%" if cov else ""
    ax.set_title(f"{row['format']}  {row['orig_w']}×{row['orig_h']}{cov_str}", fontsize=7)
    ax.axis("off")

for ax in axes[len(sample_rows):]:
    ax.axis("off")

plt.suptitle(f"Random sample — {len(sample_rows)} preprocessed images", y=1.01)
plt.tight_layout()
plt.show()

ValueError: Number of rows must be a positive integer, not 0

<Figure size 1200x0 with 0 Axes>

In [ ]:
# ── Rename files: strip old hash suffixes ({stem}_{8hex}.png → {stem}.png) ──────
import re
from pathlib import Path

RENAME_DIR = f"{DRIVE}/fetal_head_circumference/training_set/training_set_preprocessed"  # change to any directory you want to clean up

_HASH_RE = re.compile(r"^(.+)_([0-9a-f]{8})(\.[^.]+)$", re.IGNORECASE)

renamed, skipped, conflicts = 0, 0, 0
for p in sorted(Path(RENAME_DIR).iterdir()):
    m = _HASH_RE.match(p.name)
    if not m:
        skipped += 1
        continue
    new_name = m.group(1) + m.group(3)
    dst = p.parent / new_name
    if dst.exists():
        print(f"CONFLICT (skipping): {p.name} → {new_name}")
        conflicts += 1
        continue
    p.rename(dst)
    renamed += 1

print(f"Renamed : {renamed}")
print(f"Skipped (no suffix): {skipped}")
print(f"Conflicts (unchanged): {conflicts}")

Renamed : 1998
Skipped (no suffix): 1
Conflicts (unchanged): 0
